In [7]:
import pandas as pd
from scipy.spatial import cKDTree
import numpy as np
from haversine import haversine_vector, Unit

# 1. Read files
gdot = pd.read_csv("GA_supplemental/GA_fatal_crash_rates_nonzero.csv")
export = pd.read_csv("GA_supplemental/Export.csv") # from https://sigopsmetrics.dot.ga.gov/signal-info

# 2. Extract coordinates
gdot_coords = np.vstack((gdot["Lat"], gdot["Long"])).T
export_coords = np.vstack((export["latitude"], export["longitude"])).T

# 3. Build KDTree and query nearest neighbor
tree = cKDTree(export_coords)
distances_deg, indices = tree.query(gdot_coords, k=1)

# 4. Convert degree distances to meters (~111 km per degree)
approx_meters = distances_deg * 111_000  

# 5. Compute accurate geodesic distance (optional but better)
matched_export_coords = export.iloc[indices][["latitude", "longitude"]].values
accurate_meters = haversine_vector(gdot_coords, matched_export_coords, Unit.METERS)

# 6. Filter by threshold (e.g. 100 meters)
threshold_m = 500
mask = accurate_meters <= threshold_m

matched = gdot.loc[mask].copy()
matched["nearest_latitude"] = matched_export_coords[mask][:, 0]
matched["nearest_longitude"] = matched_export_coords[mask][:, 1]
matched["distance_m"] = accurate_meters[mask]

# 7. Optionally join other export columns for matched points
#matched = matched.join(export.iloc[indices[mask]].reset_index(drop=True), rsuffix="_export")
matched_export = export.iloc[indices[mask]].reset_index(drop=True)
matched = pd.concat([matched.reset_index(drop=True), matched_export.add_suffix("_export")], axis=1)


# 8. Save results
matched.to_csv("gdot_with_nearest_export_within_500m.csv", index=False)

print(f"Matched {len(matched)} of {len(gdot)} points within {threshold_m} meters.")

print("\nFinal matched dataset — columns:")
print(matched.columns.tolist())

print("\nPreview of first few rows:")
print(matched.head())



Matched 62 of 174 points within 500 meters.

Final matched dataset — columns:
['Station_ID', 'Functional_Class', 'LatLong', 'Lat', 'Long', 'AADT_2017', 'TruckPct_2017', 'AADT_2016', 'TruckPct_2016', 'AADT_2015', 'TruckPct_2015', 'AADT_2014', 'TruckPct_2014', 'AADT_2013', 'TruckPct_2013', 'AADT_2012', 'TruckPct_2012', 'AADT_2011', 'TruckPct_2011', 'AADT_2010', 'TruckPct_2010', 'AADT_2009', 'TruckPct_2009', 'AADT_2008', 'TruckPct_2008', 'ObjectId', 'geometry', 'fatal_crash_count', 'MVMT', 'crash_rate_per_100M_VMT', 'nearest_latitude', 'nearest_longitude', 'distance_m', 'signalID_export', 'zoneGroup_export', 'zone_export', 'corridor_export', 'subcorridor_export', 'agency_export', 'mainStreetName_export', 'sideStreetName_export', 'milepost_export', 'asOf_export', 'duplicate_export', 'include_export', 'modified_export', 'note_export', 'county_export', 'city_export', 'latitude_export', 'longitude_export', 'priority_export', 'classification_export']

Preview of first few rows:
  Station_ID   